# KLDM vs MatterGen lattice diffusion pipeline

This notebook compares the lattice diffusion pipeline side by side for:

- `ContinuousVPDiffusion` on the KLDM lattice representation
- `ContinuousMattergenVPDiffusion` on the MatterGen-style symmetric lattice representation

It uses one real MP20 sample and shows the main pieces of the pipeline:

1. lattice representation `x0`
2. forward sample `x_t`
3. training target / implied score
4. one reverse Euler-Maruyama step

This is a debugging notebook, not a benchmark.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "src").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT / "src"))

from kldmPlus.data import resolve_data_root
from kldmPlus.data.csp import CSPTask
from kldmPlus.data.dataset import MP20
from kldmPlus.data.transform import (
    cell_lengths_and_angles,
    lattice_spd_stats,
    vector_to_symmetric_matrix,
)
from kldmPlus.diffusionModels.continuous import (
    ContinuousMattergenVPDiffusion,
    ContinuousVPDiffusion,
)

torch.set_printoptions(precision=5, sci_mode=False)
pd.set_option("display.max_colwidth", 200)

MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


In [2]:
root = resolve_data_root(None)
sample_idx = 0
t_value = 0.35
dt = 1.0 / 1000.0
seed = 7

raw_dataset = MP20(root=root, split="train", transforms=[], download=True)
sample = raw_dataset[sample_idx]
num_atoms = torch.tensor([int(sample.num_atoms)], dtype=torch.get_default_dtype())

kldm_task = CSPTask(dataset_name="mp20", lattice_parameterization="eps", lattice_representation="kldm")
matter_task = CSPTask(dataset_name="mp20", lattice_parameterization="eps", lattice_representation="mattergen")

kldm_transform = kldm_task.make_lattice_transform(root=root, download=True)
matter_transform = matter_task.make_lattice_transform(root=root, download=True)

kldm_sample = kldm_transform(sample)
matter_sample = matter_transform(sample)

kldm_x0 = kldm_sample.l.to(torch.get_default_dtype())
matter_x0 = matter_sample.l.to(torch.get_default_dtype())

c, nu = matter_transform.stats()
kldm_diff = ContinuousVPDiffusion(parameterization="eps")
matter_diff = ContinuousMattergenVPDiffusion(c=c, nu=nu, parameterization="eps")

t = torch.tensor([t_value], dtype=torch.get_default_dtype())

print(f"repo_root={REPO_ROOT}")
print(f"data_root={root}")
print(f"sample_idx={sample_idx}")
print(f"num_atoms={int(num_atoms.item())}")
print(f"t={t_value}")
print(f"dt={dt}")
print(f"mattergen_c={c:.6f}, mattergen_nu={nu:.6f}")

repo_root=/workspace
data_root=/workspace/data
sample_idx=0
num_atoms=12
t=0.35
dt=0.001
mattergen_c=20.083511, mattergen_nu=0.648671


In [3]:
cell = sample.cell.squeeze(0)
ref_lengths, ref_angles = cell_lengths_and_angles(cell)
kldm_lengths, kldm_angles = kldm_transform.invert_to_lengths_angles(kldm_x0, num_atoms=int(num_atoms.item()))
matter_lengths, matter_angles = matter_transform.invert_to_lengths_angles(matter_x0, num_atoms=int(num_atoms.item()))

representation_df = pd.DataFrame([
    {
        "branch": "kldm",
        "x0": kldm_x0.squeeze(0).tolist(),
        "decoded_lengths": kldm_lengths.squeeze(0).tolist(),
        "decoded_angles_deg": torch.rad2deg(kldm_angles.squeeze(0)).tolist(),
    },
    {
        "branch": "mattergen",
        "x0": matter_x0.squeeze(0).tolist(),
        "decoded_lengths": matter_lengths.squeeze(0).tolist(),
        "decoded_angles_deg": torch.rad2deg(matter_angles.squeeze(0)).tolist(),
    },
    {
        "branch": "reference_cell",
        "x0": None,
        "decoded_lengths": ref_lengths.tolist(),
        "decoded_angles_deg": torch.rad2deg(ref_angles).tolist(),
    },
])
representation_df

,branch,x0,decoded_lengths,decoded_angles_deg
0,kldm,"[1.1083263158798218, 1.7293721437454224, 2.076641082763672, 0.31551799178123474, 0.1933785229921341, 0.0]","[3.0292840003967285, 5.637113571166992, 7.977627754211426]","[107.51142883300781, 100.94468688964844, 90.0]"
1,mattergen,"[2.9988255500793457, 5.546074390411377, 7.902270317077637, -0.050188466906547546, -0.42554113268852234, -1.0077738761901855]","[3.0292835235595703, 5.637115001678467, 7.9776291847229]","[107.51142120361328, 100.9447021484375, 90.0000228881836]"
2,reference_cell,None,"[3.0292840003967285, 5.637113571166992, 7.977627277374268]","[107.51142883300781, 100.94468688964844, 90.0]"


In [4]:
torch.manual_seed(seed)
shared_noise = torch.randn_like(kldm_x0)

kldm_xt, kldm_eps = kldm_diff.forward_sample(t=t, x0=kldm_x0, noise=shared_noise)
matter_xt, matter_eps = matter_diff.forward_sample(t=t, x0=matter_x0, noise=shared_noise, num_atoms=num_atoms)

forward_df = pd.DataFrame([
    {
        "branch": "kldm",
        "alpha_t": float(kldm_diff.alpha(t).item()),
        "sigma_t": float(kldm_diff.sigma(t).item()),
        "x_t": kldm_xt.squeeze(0).tolist(),
        "eps": kldm_eps.squeeze(0).tolist(),
    },
    {
        "branch": "mattergen",
        "alpha_t": float(matter_diff.alpha(t).item()),
        "sigma_t": float(matter_diff.sigma_base(t).item()),
        "x_t": matter_xt.squeeze(0).tolist(),
        "eps": matter_eps.squeeze(0).tolist(),
    },
])
forward_df

,branch,alpha_t,sigma_t,x_t,eps
0,kldm,0.534225,0.845342,"[0.4680040180683136, 1.588432788848877, 1.9097826480865479, -0.7734440565109253, 1.5326037406921387, -0.7564356327056885]","[-0.1467950940132141, 0.7861412763595581, 0.9468216300010681, -1.1143440008163452, 1.6907901763916016, -0.8948279023170471]"
1,mattergen,0.534225,0.845342,"[4.25468111038208, 7.178464889526367, 8.70639705657959, -1.8937084674835205, 2.6053004264831543, -2.0375125408172607]","[-0.1467950940132141, 0.7861412763595581, 0.9468216300010681, -1.1143440008163452, 1.6907901763916016, -0.8948279023170471]"


In [5]:
def kldm_score_from_eps(diffusion, t, pred_eps, x_shape):
    sigma_t = diffusion.sigma(t).view(-1, *([1] * (len(x_shape) - 1)))
    return -pred_eps / sigma_t.clamp_min(diffusion.eps)


def matter_score_from_eps(diffusion, t, pred_eps, num_atoms_tensor, x_shape):
    sigma_base_t = diffusion.sigma_base(t).view(-1, *([1] * (len(x_shape) - 1)))
    _, sigma_n = diffusion.mu_sigma_n(num_atoms=num_atoms_tensor, ref=pred_eps)
    sigma_n = sigma_n.view(-1, *([1] * (len(x_shape) - 1)))
    return -pred_eps / (sigma_base_t * sigma_n).clamp_min(diffusion.eps)


kldm_target = kldm_diff.training_target(t=t, x0=kldm_x0, noise=kldm_eps)
matter_target = matter_diff.training_target(t=t, x0=matter_x0, noise=matter_eps, num_atoms=num_atoms)

kldm_score = kldm_score_from_eps(kldm_diff, t, kldm_target, kldm_xt.shape)
matter_score = matter_score_from_eps(matter_diff, t, matter_target, num_atoms, matter_xt.shape)

score_df = pd.DataFrame([
    {
        "branch": "kldm",
        "training_target": kldm_target.squeeze(0).tolist(),
        "score": kldm_score.squeeze(0).tolist(),
    },
    {
        "branch": "mattergen",
        "training_target": matter_target.squeeze(0).tolist(),
        "score": matter_score.squeeze(0).tolist(),
    },
])
score_df

,branch,training_target,score
0,kldm,"[-0.1467950940132141, 0.7861412763595581, 0.9468216300010681, -1.1143440008163452, 1.6907901763916016, -0.8948279023170471]","[0.17365171015262604, -0.9299682974815369, -1.12004554271698, 1.3182166814804077, -2.0001254081726074, 1.0585395097732544]"
1,mattergen,"[-0.1467950940132141, 0.7861412763595581, 0.9468216300010681, -1.1143440008163452, 1.6907901763916016, -0.8948279023170471]","[0.08762148022651672, -0.46924498677253723, -0.565154492855072, 0.6651480197906494, -1.0092267990112305, 0.5341196656227112]"


In [6]:
torch.manual_seed(seed + 1)
kldm_prev = kldm_diff.reverse_step(t=t, x_t=kldm_xt, pred=kldm_target, dt=dt)

torch.manual_seed(seed + 1)
matter_prev = matter_diff.reverse_step(t=t, x_t=matter_xt, pred=matter_target, dt=dt, num_atoms=num_atoms)

reverse_df = pd.DataFrame([
    {
        "branch": "kldm",
        "x_t": kldm_xt.squeeze(0).tolist(),
        "x_prev": kldm_prev.squeeze(0).tolist(),
        "delta_norm": float(torch.linalg.norm((kldm_prev - kldm_xt).squeeze(0)).item()),
    },
    {
        "branch": "mattergen",
        "x_t": matter_xt.squeeze(0).tolist(),
        "x_prev": matter_prev.squeeze(0).tolist(),
        "delta_norm": float(torch.linalg.norm((matter_prev - matter_xt).squeeze(0)).item()),
    },
])
reverse_df

,branch,x_t,x_prev,delta_norm
0,kldm,"[0.4680040180683136, 1.588432788848877, 1.9097826480865479, -0.7734440565109253, 1.5326037406921387, -0.7564356327056885]","[0.49368584156036377, 1.480480432510376, 1.9508693218231201, -0.7317218780517578, 1.4701412916183472, -0.8071693181991577]",0.149056
1,mattergen,"[4.25468111038208, 7.178464889526367, 8.70639705657959, -1.8937084674835205, 2.6053004264831543, -2.0375125408172607]","[4.295348644256592, 6.956774711608887, 8.783226013183594, -1.8122965097427368, 2.4799835681915283, -2.139960289001465]",0.299217


In [7]:
matter_prior_mean = matter_diff.prior_mean(num_atoms=num_atoms, ref=matter_x0)
matter_spd = lattice_spd_stats(matter_x0)
matter_xt_spd = lattice_spd_stats(matter_xt)
matter_prev_spd = lattice_spd_stats(matter_prev)

summary = {
    "kldm": {
        "alpha_t": float(kldm_diff.alpha(t).item()),
        "sigma_t": float(kldm_diff.sigma(t).item()),
        "prior_is_standard_normal": True,
    },
    "mattergen": {
        "alpha_t": float(matter_diff.alpha(t).item()),
        "sigma_base_t": float(matter_diff.sigma_base(t).item()),
        "prior_mean": matter_prior_mean.squeeze(0).tolist(),
        "x0_spd": matter_spd,
        "xt_spd": matter_xt_spd,
        "xprev_spd": matter_prev_spd,
    },
}

print(json.dumps(summary, indent=2))

{
  "kldm": {
    "alpha_t": 0.5342254042625427,
    "sigma_t": 0.8453420400619507,
    "prior_is_standard_normal": true
  },
  "mattergen": {
    "alpha_t": 0.5342254042625427,
    "sigma_base_t": 0.8453420400619507,
    "prior_mean": [
      6.223102569580078,
      6.223102569580078,
      6.223102569580078,
      0.0,
      0.0,
      0.0
    ],
    "x0_spd": {
      "min_eig": 2.954378128051758,
      "frac_non_spd": 0.0,
      "frac_small_det": 0.0
    },
    "xt_spd": {
      "min_eig": 2.8302557468414307,
      "frac_non_spd": 0.0,
      "frac_small_det": 0.0
    },
    "xprev_spd": {
      "min_eig": 2.9822614192962646,
      "frac_non_spd": 0.0,
      "frac_small_det": 0.0
    }
  }
}


In [8]:
matter_x0_matrix = vector_to_symmetric_matrix(matter_x0.squeeze(0))
matter_xt_matrix = vector_to_symmetric_matrix(matter_xt.squeeze(0))
matter_prev_matrix = vector_to_symmetric_matrix(matter_prev.squeeze(0))

matrix_df = pd.DataFrame([
    {"stage": "mattergen_x0", "matrix": matter_x0_matrix.tolist()},
    {"stage": "mattergen_xt", "matrix": matter_xt_matrix.tolist()},
    {"stage": "mattergen_xprev", "matrix": matter_prev_matrix.tolist()},
])
matrix_df

,stage,matrix
0,mattergen_x0,"[[2.9988255500793457, -0.050188466906547546, -0.42554113268852234], [-0.050188466906547546, 5.546074390411377, -1.0077738761901855], [-0.42554113268852234, -1.0077738761901855, 7.902270317077637]]"
1,mattergen_xt,"[[4.25468111038208, -1.8937084674835205, 2.6053004264831543], [-1.8937084674835205, 7.178464889526367, -2.0375125408172607], [2.6053004264831543, -2.0375125408172607, 8.70639705657959]]"
2,mattergen_xprev,"[[4.295348644256592, -1.8122965097427368, 2.4799835681915283], [-1.8122965097427368, 6.956774711608887, -2.139960289001465], [2.4799835681915283, -2.139960289001465, 8.783226013183594]]"
